In [16]:
"""
load_to_mysql.py
================
ETL Step 3: Read prepared CSVs and load into MySQL.
Order matters due to FK: assets first, then all dependent tables.
"""

import pandas as pd
from sqlalchemy import create_engine, text


DB_HOST = "204.168.228.173"
DB_PORT = 3306
DB_NAME = "gold_market"
DB_USER = "baiastan_8ka"
DB_PASS = "baiastan2005"

ENGINE = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False
)

In [17]:
def test_connection():
    try:
        with ENGINE.connect() as conn:
            row = conn.execute(text("SELECT VERSION(), DATABASE(), NOW()")).fetchone()
            print(f"Connected: MySQL {row[0]} | db={row[1]} | server_time={row[2]}")

            tables = conn.execute(text("""
                SELECT TABLE_NAME, TABLE_ROWS
                FROM information_schema.TABLES
                WHERE TABLE_SCHEMA = :db
                ORDER BY TABLE_NAME
            """), {"db": DB_NAME}).fetchall()

            if tables:
                print(f"\nTables in '{DB_NAME}':")
                for t in tables:
                    print(f"  {t[0]:25s} (~{t[1] or 0} rows)")
            else:
                print(f"No tables in '{DB_NAME}' — DDL not run yet?")

    except Exception as e:
        print(f"Connection failed: {e}")
        raise SystemExit(1)

test_connection()

Connected: MySQL 8.4.8 | db=gold_market | server_time=2026-04-17 08:53:10

Tables in 'gold_market':
  assets                    (~14 rows)
  daily_prices              (~62775 rows)
  energy_mining             (~6480 rows)
  etf_flows                 (~10445 rows)
  macro_indicators          (~6883 rows)
  market_events             (~0 rows)


In [18]:
# ── assets reference data ─────────────────────────────────────────────────────
# 15 rows, populated once. category: precious_metal / equity_index / currency / etf / commodity

ASSETS = [
    # symbol        name                           category         data_source
    ("GC=F",        "Gold Futures",                "precious_metal", "yfinance"),
    ("SI=F",        "Silver Futures",              "precious_metal", "yfinance"),
    ("PL=F",        "Platinum Futures",            "precious_metal", "yfinance"),
    ("PA=F",        "Palladium Futures",           "precious_metal", "yfinance"),
    ("DX-Y.NYB",    "US Dollar Index (DXY)",       "currency",       "yfinance"),
    ("^GSPC",       "S&P 500 Index",               "equity_index",   "yfinance"),
    ("^VIX",        "CBOE Volatility Index",       "equity_index",   "yfinance"),
    ("EURUSD=X",    "EUR/USD Exchange Rate",       "currency",       "yfinance"),
    ("USDCNY=X",    "USD/CNY Exchange Rate",       "currency",       "yfinance"),
    ("RUB=X",       "USD/RUB Exchange Rate",       "currency",       "yfinance"),
    ("GLD",         "SPDR Gold ETF",               "etf",            "yfinance"),
    ("IAU",         "iShares Gold ETF",            "etf",            "yfinance"),
    ("CL=F",        "WTI Crude Oil Futures",       "commodity",      "yfinance"),
    ("BZ=F",        "Brent Crude Oil Futures",     "commodity",      "yfinance"),
    ("HG=F",        "Copper Futures",              "commodity",      "yfinance"),
]


def load_assets(conn) -> None:
    """Insert assets reference rows. INSERT IGNORE skips existing symbols."""
    print("  Loading assets...")
    df = pd.DataFrame(ASSETS, columns=["symbol", "name", "category", "data_source"])
    for _, row in df.iterrows():
        conn.execute(text("""
            INSERT IGNORE INTO assets (symbol, name, category, data_source)
            VALUES (:symbol, :name, :category, :data_source)
        """), row.to_dict())
    count = conn.execute(text("SELECT COUNT(*) FROM assets")).scalar()
    print(f"    assets: {count} rows")


def get_asset_id_map(conn) -> dict:
    """Return {symbol: asset_id} for FK lookups."""
    rows = conn.execute(text("SELECT symbol, asset_id FROM assets"))
    return {r[0]: r[1] for r in rows}


def _get_max_date(conn, table: str, date_col: str):
    """Return the latest date already in a table, or None if empty."""
    return conn.execute(text(f"SELECT MAX({date_col}) FROM {table}")).scalar()


def load_daily_prices(conn, asset_id_map: dict) -> None:
    """Insert only rows newer than MAX(price_date) already in the table."""
    print("  Loading daily_prices...")
    df = pd.read_csv("table_daily_prices.csv", parse_dates=["price_date"])

    last = _get_max_date(conn, "daily_prices", "price_date")
    if last:
        before = len(df)
        df = df[df["price_date"] > pd.Timestamp(last)]
        print(f"    last in DB: {last} | CSV rows: {before} | new: {len(df)}")

    if df.empty:
        print("    daily_prices: nothing new")
        return

    df["asset_id"] = df["symbol"].map(asset_id_map)
    df = df.dropna(subset=["asset_id"])
    df["asset_id"] = df["asset_id"].astype(int)
    df[["asset_id", "price_date", "close_price"]].to_sql(
        "daily_prices", conn, if_exists="append", index=False, method="multi", chunksize=1000
    )
    count = conn.execute(text("SELECT COUNT(*) FROM daily_prices")).scalar()
    print(f"    daily_prices: {count} rows total")


def load_macro_indicators(conn) -> None:
    """Insert only rows newer than MAX(indicator_date) already in the table."""
    print("  Loading macro_indicators...")
    df = pd.read_csv("table_macro_indicators.csv", parse_dates=["indicator_date"])

    last = _get_max_date(conn, "macro_indicators", "indicator_date")
    if last:
        before = len(df)
        df = df[df["indicator_date"] > pd.Timestamp(last)]
        print(f"    last in DB: {last} | CSV rows: {before} | new: {len(df)}")

    if df.empty:
        print("    macro_indicators: nothing new")
        return

    df.to_sql("macro_indicators", conn, if_exists="append", index=False, method="multi", chunksize=500)
    count = conn.execute(text("SELECT COUNT(*) FROM macro_indicators")).scalar()
    print(f"    macro_indicators: {count} rows total")


def load_etf_flows(conn, asset_id_map: dict) -> None:
    """Insert only rows newer than MAX(flow_date) already in the table."""
    print("  Loading etf_flows...")
    df = pd.read_csv("table_etf_flows.csv", parse_dates=["flow_date"])

    last = _get_max_date(conn, "etf_flows", "flow_date")
    if last:
        before = len(df)
        df = df[df["flow_date"] > pd.Timestamp(last)]
        print(f"    last in DB: {last} | CSV rows: {before} | new: {len(df)}")

    if df.empty:
        print("    etf_flows: nothing new")
        return

    df["asset_id"] = df["symbol"].map(asset_id_map)
    df = df.dropna(subset=["asset_id"])
    df["asset_id"] = df["asset_id"].astype(int)
    df[["asset_id", "flow_date", "etf_price", "gld_gold_premium", "etf_signal"]].to_sql(
        "etf_flows", conn, if_exists="append", index=False, method="multi", chunksize=500
    )
    count = conn.execute(text("SELECT COUNT(*) FROM etf_flows")).scalar()
    print(f"    etf_flows: {count} rows total")


def load_energy_mining(conn) -> None:
    """Insert only rows newer than MAX(record_date) already in the table."""
    print("  Loading energy_mining...")
    df = pd.read_csv("table_energy_mining.csv", parse_dates=["record_date"])

    last = _get_max_date(conn, "energy_mining", "record_date")
    if last:
        before = len(df)
        df = df[df["record_date"] > pd.Timestamp(last)]
        print(f"    last in DB: {last} | CSV rows: {before} | new: {len(df)}")

    if df.empty:
        print("    energy_mining: nothing new")
        return

    df.to_sql("energy_mining", conn, if_exists="append", index=False, method="multi", chunksize=500)
    count = conn.execute(text("SELECT COUNT(*) FROM energy_mining")).scalar()
    print(f"    energy_mining: {count} rows total")


def load_to_mysql() -> None:
    """Run full Step 3. assets must be loaded first (FK dependency)."""
    print(f"=== load_to_mysql | target: {DB_HOST}/{DB_NAME} ===\n")

    with ENGINE.begin() as conn:
        load_assets(conn)
        asset_id_map = get_asset_id_map(conn)
        print(f"  asset_id_map: {len(asset_id_map)} assets\n")

        load_daily_prices(conn, asset_id_map)
        load_macro_indicators(conn)
        load_etf_flows(conn, asset_id_map)
        load_energy_mining(conn)

    print("\nDone. market_events must be populated manually via SQL.")


load_to_mysql()

=== load_to_mysql | target: 204.168.228.173/gold_market ===

  Loading assets...
    assets: 15 rows
  asset_id_map: 15 assets

  Loading daily_prices...
    last in DB: 2026-04-16 | CSV rows: 62499 | new: 0
    daily_prices: nothing new
  Loading macro_indicators...
    last in DB: 2026-04-16 | CSV rows: 6982 | new: 0
    macro_indicators: nothing new
  Loading etf_flows...
    last in DB: 2026-04-16 00:00:00 | CSV rows: 10722 | new: 0
    etf_flows: nothing new
  Loading energy_mining...
    energy_mining: 6429 rows total

Done. market_events must be populated manually via SQL.
